# MAE — Masked Autoencoders Are Scalable Vision Learners (2021)

_Papers · Self-supervised & Representation_

**Hide 75% of an image's patches, ask a network to paint them back, and the encoder learns powerful features for free.**

---

This notebook is a practice scaffold for the **MAE — Masked Autoencoders Are Scalable Vision Learners (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Sanity-check the masked-patch loss

MAE's loss is the mean squared pixel error over the **masked** patches only — visible patches never enter it. We make this concrete with a tiny worked example: `N=4` patches at mask ratio `r=0.75` leaves 1 visible patch and 3 masked. We average the squared errors of the three masked patches to get the reported reconstruction MSE.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity-check the worked example.
# N=4 patches, mask ratio r=0.75  ->  N_vis = (1-r)*N = 1 visible patch, 3 masked.
# Loss = mean over MASKED patches of squared pixel error (visible patches never enter the loss).
N, r = 4, 0.75
N_vis = round((1 - r) * N)
print("worked example:  N_vis =", N_vis)   # 1

# Three masked patches: prediction p vs target x.
xA, pA = torch.tensor([1.0, 0.0]), torch.tensor([0.5, 0.0])   # masked patch A
xB, pB = torch.tensor([0.0, 2.0]), torch.tensor([0.0, 1.0])   # masked patch B
xC, pC = torch.tensor([1.0, 1.0]), torch.tensor([1.0, 1.0])   # masked patch C (perfect)

errs = [((pA - xA) ** 2).sum(), ((pB - xB) ** 2).sum(), ((pC - xC) ** 2).sum()]   # 0.25, 1.0, 0.0
L = torch.stack(errs).mean()
print("worked example:  masked-patch MSE =", round(L.item(), 4))   # 0.4167

### Step 2 — Patchify, then define the asymmetric MAE

We split each 28x28 MNIST image into a 4x4 grid of 7x7 patches (16 patches of 49 pixels). The key MAE idea is **asymmetry**: the encoder processes only the *visible* patches (no mask tokens), which is what makes it fast. The decoder then re-assembles a full-length sequence by inserting a single shared learned **mask token** into the blanks and reconstructs the pixels. The loss is applied to masked patches only.

In [ ]:
# Patchify / unpatchify for 28x28 single-channel images, patch size P=7  ->  N=16 patches.
P, IMG, C = 7, 28, 1
GRID = IMG // P   # 4x4 grid
N = GRID * GRID   # 16 patches
PD = P * P * C    # 49 pixels per patch

def patchify(x):                      # x:(B,1,28,28) -> (B, N, PD)
    B = x.shape[0]
    x = x.unfold(2, P, P).unfold(3, P, P)            # (B,1,GRID,GRID,P,P)
    x = x.permute(0, 2, 3, 1, 4, 5).reshape(B, N, PD)
    return x

def unpatchify(p):                    # (B,N,PD) -> (B,1,28,28)
    B = p.shape[0]
    p = p.reshape(B, GRID, GRID, C, P, P).permute(0, 3, 1, 4, 2, 5)
    return p.reshape(B, C, IMG, IMG)


# The asymmetric MAE: encode VISIBLE patches only; decode with a shared learned mask token.
class TinyMAE(nn.Module):
    def __init__(self, dim=64, dec_dim=32, enc_depth=3, dec_depth=2, heads=4):
        super().__init__()
        self.embed   = nn.Linear(PD, dim)                            # patch -> token
        self.pos     = nn.Parameter(torch.zeros(1, N, dim))          # encoder position embeddings
        enc = nn.TransformerEncoderLayer(dim, heads, dim*2, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(enc, enc_depth)
        self.enc2dec = nn.Linear(dim, dec_dim)                       # bridge to the lighter decoder
        self.mask_tok = nn.Parameter(torch.zeros(1, 1, dec_dim))     # ONE shared learned mask token
        self.dec_pos = nn.Parameter(torch.zeros(1, N, dec_dim))      # decoder position embeddings
        dec = nn.TransformerEncoderLayer(dec_dim, heads, dec_dim*2, batch_first=True, dropout=0.0)
        self.decoder = nn.TransformerEncoder(dec, dec_depth)
        self.head    = nn.Linear(dec_dim, PD)                        # token -> P*P pixels

    def random_mask(self, B, ratio):
        n_vis = max(1, round((1 - ratio) * N))
        # per-sample random permutation; first n_vis indices are VISIBLE, the rest are masked.
        noise = torch.rand(B, N, device=device)
        ids_shuffle = noise.argsort(dim=1)                           # shuffle
        ids_restore = ids_shuffle.argsort(dim=1)                     # inverse permutation
        vis_ids = ids_shuffle[:, :n_vis]                             # which patches are visible
        return vis_ids, ids_restore, n_vis

    def forward(self, x, ratio=0.75):
        B = x.shape[0]
        tokens = self.embed(patchify(x)) + self.pos                 # (B,N,dim)
        vis_ids, ids_restore, n_vis = self.random_mask(B, ratio)
        # ENCODER sees VISIBLE patches only (no mask tokens) -- the asymmetry + the speed-up.
        vis = torch.gather(tokens, 1, vis_ids[:, :, None].expand(-1, -1, tokens.shape[2]))
        enc = self.enc2dec(self.encoder(vis))                       # (B, n_vis, dec_dim)
        # Re-assemble full length-N sequence: encoded visible tokens + shared mask token in the blanks.
        mask = self.mask_tok.expand(B, N - n_vis, -1)
        full = torch.cat([enc, mask], dim=1)                        # still in shuffled order
        full = torch.gather(full, 1, ids_restore[:, :, None].expand(-1, -1, enc.shape[2]))
        dec = self.decoder(full + self.dec_pos)                     # DECODER sees full sequence
        pred = self.head(dec)                                       # (B,N,PD) predicted pixels
        # boolean mask of which patches were MASKED (True = masked), via restore order
        is_masked = torch.ones(B, N, device=device)
        is_masked.scatter_(1, vis_ids, 0.0)                         # 0 at visible, 1 at masked
        return pred, is_masked.bool()

    def encode_full(self, x):          # for the linear probe: encode the WHOLE image (no masking)
        tokens = self.embed(patchify(x)) + self.pos
        return self.encoder(tokens).mean(dim=1)                     # mean-pool token features


def mae_loss(pred, target, is_masked):
    # MSE per patch, then average over MASKED patches only (the formula).
    per_patch = ((pred - target) ** 2).mean(dim=-1)                 # (B,N)
    return (per_patch * is_masked).sum() / is_masked.sum()

### Step 3 — Load MNIST and pretrain the encoder

We take a small MNIST subset (4000 train / 1000 test) and pretrain the MAE at the paper's default 75% masking. The training target for each image is its own patches — the network learns to reconstruct the hidden 75% from the visible 25%. `linear_probe` then freezes the encoder and trains a single linear classifier on its mean-pooled features, the standard way to measure representation quality.

In [ ]:
# Data: a small MNIST subset (torchvision, preinstalled in Colab).
tf = T.ToTensor()
train = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
test  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)

ti = np.random.RandomState(0).permutation(len(train))[:4000]
Xtr = torch.stack([train[i][0] for i in ti]).to(device)
ytr = torch.tensor([train[i][1] for i in ti]).to(device)

vi = np.random.RandomState(1).permutation(len(test))[:1000]
Xte = torch.stack([test[i][0] for i in vi]).to(device)
yte = torch.tensor([test[i][1] for i in vi]).to(device)


# Pretrain MAE at a given mask ratio; return the trained model.
def pretrain(ratio=0.75, epochs=12, B=256):
    torch.manual_seed(0)
    m = TinyMAE().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        nb = 0
        for s in range(0, len(Xtr), B):
            xb = Xtr[perm[s:s+B]]
            target = patchify(xb)
            pred, is_masked = m(xb, ratio=ratio)
            loss = mae_loss(pred, target, is_masked)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
            nb += 1
        if ep % 4 == 0:
            print(f"  [r={ratio:.2f}] ep {ep:2d}  recon MSE {tot/nb:.4f}")
    return m


# Linear probe: freeze the encoder, train ONE linear classifier on its features.
def linear_probe(m, epochs=40):
    m.eval()
    with torch.no_grad():
        Ftr, Fte = m.encode_full(Xtr), m.encode_full(Xte)
    clf = nn.Linear(Ftr.shape[1], 10).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2)
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(clf(Ftr), ytr).backward()
        opt.step()
    return (clf(Fte).argmax(1) == yte).float().mean().item()

### Step 4 — Reconstruct, then probe

Finally we run pretraining and inspect the result on one held-out image. We report the reconstruction error separately on masked vs visible patches — the masked error is the real test (visible patches are not in the loss). Then the linear probe shows that a single frozen-feature classifier already separates the digits, the payoff of self-supervised pretraining.

In [ ]:
print("\n=== Pretrain MAE at the paper's default 75% masking ===")
mae = pretrain(ratio=0.75)

# Reconstruct an image from its 25% visible patches; report error on masked vs visible patches.
mae.eval()
with torch.no_grad():
    x = Xte[:1]
    target = patchify(x)
    pred, is_masked = mae(x, ratio=0.75)
    per_patch = ((pred - target) ** 2).mean(dim=-1)[0]
    print("masked-patch recon MSE :", round(per_patch[is_masked[0]].mean().item(), 4))
    print("visible-patch recon MSE:", round(per_patch[~is_masked[0]].mean().item(), 4),
          " (visible patches are NOT in the loss)")

print("linear-probe accuracy @ r=0.75:", round(linear_probe(mae), 4))
# Our small run, not the paper's numbers. The encoder reconstructs the hidden 75% from the visible
# 25%, and a single frozen-feature linear classifier already separates the digits.

## Visualize the data & results

_What mask ratio gives the best features? Sweep r, then linear-probe the frozen encoder (and watch reconstruction error)._

### Step 5 — Rebuild a compact MAE and the data

The visualization is self-contained (its own imports and a slimmer `TinyMAE`) so it runs independently. This first part defines the patchify helper, the model, the masked-patch loss, and loads the same MNIST subset — everything needed before we sweep the mask ratio.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

P, IMG, C = 7, 28, 1
GRID = IMG // P
N = GRID * GRID
PD = P * P * C   # 16 patches of 49 pixels

def patchify(x):
    B = x.shape[0]
    x = x.unfold(2, P, P).unfold(3, P, P)
    return x.permute(0, 2, 3, 1, 4, 5).reshape(B, N, PD)

class TinyMAE(nn.Module):
    def __init__(s, dim=64, dd=32, ed=3, dc=2, h=4):
        super().__init__()
        s.embed = nn.Linear(PD, dim)
        s.pos = nn.Parameter(torch.zeros(1, N, dim))
        s.encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(dim, h, dim*2, batch_first=True, dropout=0.0), ed)
        s.enc2dec = nn.Linear(dim, dd)
        s.mask_tok = nn.Parameter(torch.zeros(1, 1, dd))
        s.dec_pos = nn.Parameter(torch.zeros(1, N, dd))
        s.decoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(dd, h, dd*2, batch_first=True, dropout=0.0), dc)
        s.head = nn.Linear(dd, PD)

    def forward(s, x, ratio):
        B = x.shape[0]
        tok = s.embed(patchify(x)) + s.pos
        nv = max(1, round((1 - ratio) * N))
        ids = torch.rand(B, N, device=device).argsort(1)
        restore = ids.argsort(1)
        vis = ids[:, :nv]
        v = torch.gather(tok, 1, vis[:, :, None].expand(-1, -1, tok.shape[2]))
        e = s.enc2dec(s.encoder(v))
        mk = s.mask_tok.expand(B, N - nv, -1)
        full = torch.cat([e, mk], 1)
        full = torch.gather(full, 1, restore[:, :, None].expand(-1, -1, e.shape[2]))
        pred = s.head(s.decoder(full + s.dec_pos))
        m = torch.ones(B, N, device=device)
        m.scatter_(1, vis, 0.0)   # 1 = masked
        return pred, m.bool()

    def encode_full(s, x):
        return s.encoder(s.embed(patchify(x)) + s.pos).mean(1)

def mae_loss(pred, tgt, msk):
    pp = ((pred - tgt) ** 2).mean(-1)
    return (pp * msk).sum() / msk.sum()   # MSE on MASKED patches only

tf = T.ToTensor()
tr = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
te = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
ti = np.random.RandomState(0).permutation(len(tr))[:4000]
Xtr = torch.stack([tr[i][0] for i in ti]).to(device)
ytr = torch.tensor([tr[i][1] for i in ti]).to(device)
vi = np.random.RandomState(1).permutation(len(te))[:1000]
Xte = torch.stack([te[i][0] for i in vi]).to(device)
yte = torch.tensor([te[i][1] for i in vi]).to(device)

### Step 6 — Sweep the mask ratio and probe each encoder

Now we run the actual experiment: pretrain at several mask ratios from 0.25 to 0.90, and after each one freeze the encoder and linear-probe it. Watch the two numbers move in opposite ways — reconstruction MSE just rises with the ratio (harder to fill more blanks), but probe accuracy peaks at a high ratio (~0.75), the paper's surprising finding that masking *most* of the image gives the strongest features.

In [ ]:
def run(ratio, epochs=12, B=256):
    torch.manual_seed(0)
    m = TinyMAE().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    last = 0.0
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        nb = 0
        for s0 in range(0, len(Xtr), B):
            xb = Xtr[perm[s0:s0+B]]
            pred, msk = m(xb, ratio)
            l = mae_loss(pred, patchify(xb), msk)
            opt.zero_grad()
            l.backward()
            opt.step()
            tot += l.item()
            nb += 1
        last = tot / nb
    m.eval()
    with torch.no_grad():
        Ftr, Fte = m.encode_full(Xtr), m.encode_full(Xte)
    clf = nn.Linear(Ftr.shape[1], 10).to(device)
    o = torch.optim.Adam(clf.parameters(), lr=1e-2)
    for _ in range(40):
        o.zero_grad()
        F.cross_entropy(clf(Ftr), ytr).backward()
        o.step()
    acc = (clf(Fte).argmax(1) == yte).float().mean().item()
    return round(last, 4), round(acc, 4)

for r in [0.25, 0.50, 0.65, 0.75, 0.85, 0.90]:
    mse, acc = run(r)
    print(f"r={r:.2f}  recon_MSE={mse}  probe_acc={acc}")
# probe_acc peaks at a high ratio (~0.75); recon_MSE just rises with r. Our small run, not the paper's.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Visible-patch count + loss. An image is cut into $N=16$ patches and you use mask ratio
            $r=0.75$. (a) How many patches does the encoder process? (b) The decoder predicts pixels for the
            masked patches; three of them have squared pixel errors $0.4$, $0.9$, and $0.2$, and these three
            errors average to the whole masked set's error. What reconstruction MSE does the loss report — and
            would including the (perfectly-copied) visible patches raise or lower this number?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the visible count: $N_{\text{vis}} = (1-r)\,N = 0.25 \times 16 = 4$. — _The encoder sees only the visible patches; at 75% masking that is a quarter of $N$._
- Average the masked-patch errors: $\tfrac{1}{3}(0.4 + 0.9 + 0.2) = \tfrac{1.5}{3} = 0.5$. — _The loss is the mean squared error over the masked set $\mathcal{M}$ only._
- Note visible patches are excluded by design; were they included (with ~0 error from copying), they would pull the average DOWN toward 0. — _Copied visible pixels have near-zero error, so averaging them in would understate the real prediction difficulty — exactly why MAE excludes them._

**Answer:** (a) The encoder processes $N_{\text{vis}} = (1-0.75)\times 16 = \mathbf{4}$ patches. (b) The
                 reported reconstruction MSE is $\tfrac{1}{3}(0.4+0.9+0.2) = \mathbf{0.5}$, averaged over the
                 masked patches only. Including the visible patches (which the network copies at ~zero error)
                 would lower the number artificially — that is precisely why the MAE loss is restricted
                 to $\mathcal{M}$: it should measure how well the network guessed the unknown, not how well it
                 copied the known.

</details>

**Problem 2.** The mask-ratio ablation (Figure 5). You pretrain MAE at mask ratios $r = 0.25, 0.50, 0.75,
            0.90$, then linear-probe the frozen encoder each time. The probe accuracies come out (roughly)
            $0.62, 0.71, 0.78, 0.66$. What ratio is best, what shape is the curve, and what does this tell you
            about images as a self-supervised signal?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the maximum: probe accuracy peaks at $r=0.75$ (0.78), higher than the low-mask runs. — _A higher mask ratio than the naive "mask a little" choice gives the best features — the paper's "surprisingly high" optimum._
- Note the drop from $0.75$ to $0.90$ (0.78 -> 0.66). — _If you mask too much, too little context remains to reconstruct meaningfully, so the features degrade again — the curve is an inverted U._
- Connect to redundancy: a low ratio is solvable by interpolation; a very high one removes too much; the sweet spot forces global reasoning. — _Images are redundant, so the task is only "nontrivial and meaningful" (abstract) when enough is hidden — but not everything._

**Answer:** The best ratio is $r=0.75$, and the curve is an inverted U: probe accuracy rises
                 from 0.25 up to a peak near 0.75, then falls by 0.90. This matches the paper's Figure 5 finding
                 that "the optimal ratios are surprisingly high." The lesson: because images are redundant,
                 masking only a little leaves a task solvable by copying neighbours (weak features), while masking
                 almost everything leaves too little to reason from — so a high-but-not-total mask (~75%) is the
                 hardest useful task and yields the strongest encoder. (Our numbers are a small run, not
                 the paper's.)

</details>

**Problem 3.** A teammate says: "MAE is just BERT — mask 15% of the tokens and predict them. Let's reuse our BERT
            mask ratio and feed the masked image (with grey blanks) straight through the encoder." Name the two
            things wrong with this plan and what MAE does instead.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot the wrong mask ratio: 15% is BERT's number for text; images are far more redundant and need ~75%. — _At 15% masking, image blanks are guessable by interpolation, so the encoder learns little (§1, §3)._
- Spot the wrong encoder input: feeding the full masked image (blanks included) puts mask tokens INTO the encoder. — _MAE's asymmetry requires the encoder to see visible patches only; mask tokens enter at the decoder, not the encoder._
- State MAE's recipe: encode the 25% visible patches; reassemble with a shared mask token in the blanks; decode with a small decoder; loss on masked patches only. — _This is what makes the task hard (high ratio) and training fast (short encoder sequence) at the same time._

**Answer:** Two errors. (1) The ratio: 15% is BERT's text setting, but images are redundant, so a
                 masked patch is easily interpolated — MAE masks a far higher 75% to make the task
                 nontrivial. (2) The encoder input: feeding the full greyed image pushes mask tokens
                 through the encoder, which is exactly what MAE avoids. MAE's asymmetric design encodes
                 only the visible 25% of patches (no mask tokens), then inserts a shared learned mask
                 token into the blanks at the decoder, a small network that reconstructs pixels with the
                 loss on masked patches only. That gives a harder task and ~3x faster training together.

</details>